# Bone Fracture Point RL Pipeline

Run order:
1. Install dependencies if needed.
2. Load `.env.staging` and confirm the staged Moondream keys.
3. Build the collapsed point dataset from `raw_dataset/bone fracture.coco`.
4. Inspect split balance, empty rows, and preserved raw source labels.
5. Launch all 3 current point reruns in parallel.
6. Check status or wait for all point runs to finish.
7. Launch the current detect probe.
8. Check status or wait for the detect probe to finish.
9. Optionally push the built dataset to HF.


In [86]:
from pathlib import Path

print(Path.cwd().resolve())


/Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture


In [87]:
# Install once if needed.
!python -m pip install -q datasets pillow numpy scipy wandb python-dotenv


## 1) Load `.env.staging`

The local build path does not need a Roboflow key. The current point reruns use `CICID_GPUB_MOONDREAM_API_KEY_1..3`, and the current detect probe uses `CICID_GPUB_MOONDREAM_API_KEY_4`.


In [88]:
import os
from pathlib import Path
from dotenv import load_dotenv

loaded_files = []
for env_name, override in ((".env.staging", True), (".env", False)):
    env_path = Path(env_name)
    if env_path.exists():
        load_dotenv(env_path, override=override)
        loaded_files.append(env_name)

print("loaded env files:", loaded_files)
for key in [
    "CICID_GPUB_MOONDREAM_API_KEY_1",
    "CICID_GPUB_MOONDREAM_API_KEY_2",
    "CICID_GPUB_MOONDREAM_API_KEY_3",
    "CICID_GPUB_MOONDREAM_API_KEY_4",
    "HUGGINGFACE_HUB_TOKEN",
    "HF_TOKEN",
    "ROBOFLOW_API_KEY",
]:
    print(f"{key}:", "set" if os.getenv(key) else "MISSING")


loaded env files: ['.env.staging']
CICID_GPUB_MOONDREAM_API_KEY_1: set
CICID_GPUB_MOONDREAM_API_KEY_2: set
CICID_GPUB_MOONDREAM_API_KEY_3: set
CICID_GPUB_MOONDREAM_API_KEY_4: set
HUGGINGFACE_HUB_TOKEN: set
HF_TOKEN: MISSING
ROBOFLOW_API_KEY: set


## 2) Build the local point dataset

This collapses all annotated abnormalities into the single target `bone fracture or abnormality` and writes the normalized dataset to `outputs/maxs-m87_bone_fracture_point_v1`.


In [89]:
!python build_bone_fracture_hf_dataset.py   --config configs/build_bone_fracture_hf_dataset_default.json   --env-file .env.staging   --push-to-hub ""


using local COCO export at /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture/raw_dataset/bone fracture.coco
Saving the dataset (1/1 shards): 100%|█| 369/369 [00:00<00:00, 3504.64 examples/
Saving the dataset (1/1 shards): 100%|█| 44/44 [00:00<00:00, 3300.30 examples/s]
saved normalized dataset to /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture/outputs/maxs-m87_bone_fracture_point_v1 (train=369, validation=45, test=44)


## 3) Inspect the built dataset

This checks split sizes, positive vs empty rows, and preserved raw source labels so you can catch dataset issues before training.


In [90]:
import json
from collections import Counter
from pathlib import Path
from datasets import load_from_disk

output_dir = Path("outputs/maxs-m87_bone_fracture_point_v1")
metadata = json.loads((output_dir / "metadata.json").read_text())
stats = json.loads((output_dir / "stats.json").read_text())
ds = load_from_disk(str(output_dir))

print("output_dir:", output_dir.resolve())
print("split_sizes:", stats["split_sizes"])
print("positive_row_counts:", stats["positive_row_counts"])
print("empty_row_counts:", stats["empty_row_counts"])
print("raw_source_label_catalog:", stats["raw_source_label_catalog"])
print("raw_source_label_counts:")
for split_name, counts in stats["raw_source_label_counts"].items():
    print(" ", split_name, counts)

for split_name in ds:
    class_counts = Counter()
    for row in ds[split_name]:
        for item in json.loads(row["answer_boxes"]):
            class_counts[item["class_name"]] += 1
    print(split_name, "collapsed target counts:", dict(class_counts))

print("\nexample positive row:")
for row in ds["train"]:
    if int(row["class_count"]) > 0:
        print({k: row[k] for k in ["source_split", "source_image_id", "source_base_id", "class_count"]})
        print(json.loads(row["answer_boxes"])[:2])
        break

print("\nexample empty row:")
empty_example = None
for split_name in ds:
    for row in ds[split_name]:
        if int(row["class_count"]) == 0:
            empty_example = (
                split_name,
                {k: row[k] for k in ["source_split", "source_image_id", "source_base_id", "class_count"]},
                json.loads(row["answer_boxes"]),
            )
            break
    if empty_example is not None:
        break

if empty_example is None:
    print("no empty rows found")
else:
    split_name, row_info, boxes = empty_example
    print(split_name, row_info)
    print(boxes)


output_dir: /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture/outputs/maxs-m87_bone_fracture_point_v1
split_sizes: {'test': 44, 'train': 369, 'validation': 45}
positive_row_counts: {'test': 43, 'train': 352, 'validation': 42}
empty_row_counts: {'test': 1, 'train': 17, 'validation': 3}
raw_source_label_catalog: ['angle', 'fracture', 'line', 'messed_up_angle']
raw_source_label_counts:
  test {'angle': 4, 'fracture': 24, 'line': 16, 'messed_up_angle': 8}
  train {'angle': 30, 'fracture': 271, 'line': 134, 'messed_up_angle': 55}
  validation {'angle': 7, 'fracture': 31, 'line': 14, 'messed_up_angle': 7}
train collapsed target counts: {'bone fracture or abnormality': 490}
validation collapsed target counts: {'bone fracture or abnormality': 59}
test collapsed target counts: {'bone fracture or abnormality': 52}

example positive row:
{'source_split': 'train', 'source_image_id': '204_jpg.rf.ewM0XMas1Yc76SI5jkrf', 'source_base_id': '204_jpg', 'class_count': 2}
[{'x_min': 0.

## 4) Run Current Point Reruns in Parallel

This launches the three current bone-fracture point reruns in parallel. Each run writes to its own log file under `logs/current_parallel/`.

The configs rotate across staging API keys:
- control short KL guard: `CICID_GPUB_MOONDREAM_API_KEY_1`
- recall primary short KL guard: `CICID_GPUB_MOONDREAM_API_KEY_2`
- recall off-policy lite KL guard: `CICID_GPUB_MOONDREAM_API_KEY_3`


In [91]:
import subprocess
import sys
from pathlib import Path

parallel_configs = [
    "configs/cicd/cicd_train_bone_fracture_point_control_short_klguard.json",
    "configs/cicd/cicd_train_bone_fracture_point_recall_primary_short_klguard.json",
    "configs/cicd/cicd_train_bone_fracture_point_recall_offpolicy_lite_klguard.json",
]

logs_dir = Path("logs/current_parallel")
logs_dir.mkdir(parents=True, exist_ok=True)

parallel_train_runs = []
for config_path in parallel_configs:
    config_name = Path(config_path).stem
    log_path = logs_dir / f"{config_name}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    cmd = [
        sys.executable,
        "train_bone_fracture_point.py",
        "--config",
        config_path,
    ]
    proc = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT)
    parallel_train_runs.append(
        {
            "name": config_name,
            "config_path": config_path,
            "pid": proc.pid,
            "process": proc,
            "log_path": str(log_path),
            "log_handle": log_handle,
        }
    )
    print(f"started {config_name} | pid={proc.pid} | log={log_path}")

print(f"launched {len(parallel_train_runs)} parallel training runs")


started cicd_train_bone_fracture_point_control_short_klguard | pid=29371 | log=logs/current_parallel/cicd_train_bone_fracture_point_control_short_klguard.log
started cicd_train_bone_fracture_point_recall_primary_short_klguard | pid=29372 | log=logs/current_parallel/cicd_train_bone_fracture_point_recall_primary_short_klguard.log
started cicd_train_bone_fracture_point_recall_offpolicy_lite_klguard | pid=29373 | log=logs/current_parallel/cicd_train_bone_fracture_point_recall_offpolicy_lite_klguard.log
launched 3 parallel training runs


## 5) Check Parallel Run Status and Tail Recent Logs

Run this while the current point reruns are active to see return codes and the last few log lines for each process.


In [92]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    proc = run["process"]
    return_code = proc.poll()
    status = "running" if return_code is None else f"finished rc={return_code}"
    print(f"\n[{run['name']}] pid={run['pid']} status={status}")
    log_path = Path(run["log_path"])
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in lines[-10:]:
            print(line)



[cicd_train_bone_fracture_point_control_short_klguard] pid=29371 status=running

[cicd_train_bone_fracture_point_recall_primary_short_klguard] pid=29372 status=running

[cicd_train_bone_fracture_point_recall_offpolicy_lite_klguard] pid=29373 status=running


## 6) Wait For All Parallel Runs To Finish

This blocks until every launched process exits, then closes the log handles.


In [93]:
if "parallel_train_runs" not in globals() or not parallel_train_runs:
    raise RuntimeError("Launch the parallel runs first.")

for run in parallel_train_runs:
    rc = run["process"].wait()
    run["log_handle"].close()
    print(f"finished {run['name']} with rc={rc}")


finished cicd_train_bone_fracture_point_control_short_klguard with rc=0
finished cicd_train_bone_fracture_point_recall_primary_short_klguard with rc=0
finished cicd_train_bone_fracture_point_recall_offpolicy_lite_klguard with rc=0


## 8) Run The Current Detect Probe

This uses the existing detect dataset at `outputs/maxs-m87_bone_fracture_detect_v1` and launches the current fracture-only detect probe. The run writes to its own log file under `logs/current_detect/`.

The config uses staging API key `CICID_GPUB_MOONDREAM_API_KEY_4`.


In [94]:
import subprocess
import sys
from pathlib import Path

detect_dataset_path = Path("outputs/maxs-m87_bone_fracture_detect_v1")
if not detect_dataset_path.exists():
    raise FileNotFoundError(
        "Detect dataset not found at outputs/maxs-m87_bone_fracture_detect_v1. "
        "Build it separately or point this cell at the correct detect dataset path."
    )

current_detect_configs = [
    "configs/cicd/cicd_train_bone_fracture_detect_fracture_only_f1_klguard.json",
]

detect_logs_dir = Path("logs/current_detect")
detect_logs_dir.mkdir(parents=True, exist_ok=True)

current_detect_runs = []
for config_path in current_detect_configs:
    config_name = Path(config_path).stem
    log_path = detect_logs_dir / f"{config_name}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    cmd = [
        sys.executable,
        "train_bone_fracture_detect.py",
        "--config",
        config_path,
        "--env-file",
        ".env.staging",
        "--dataset-path",
        str(detect_dataset_path),
    ]
    proc = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT)
    current_detect_runs.append(
        {
            "name": config_name,
            "config_path": config_path,
            "pid": proc.pid,
            "process": proc,
            "log_path": str(log_path),
            "log_handle": log_handle,
        }
    )
    print(f"started {config_name} | pid={proc.pid} | log={log_path}")

print(f"launched {len(current_detect_runs)} current detect runs")


started cicd_train_bone_fracture_detect_fracture_only_f1_klguard | pid=72973 | log=logs/current_detect/cicd_train_bone_fracture_detect_fracture_only_f1_klguard.log
launched 1 current detect runs


## 9) Check Current Detect Probe Status and Tail Recent Logs

Run this while the current detect probe is active to see the return code and the last few log lines.


In [95]:
if "current_detect_runs" not in globals() or not current_detect_runs:
    raise RuntimeError("Launch the current detect probe first.")

for run in current_detect_runs:
    proc = run["process"]
    return_code = proc.poll()
    status = "running" if return_code is None else f"finished rc={return_code}"
    print(f"\n[{run['name']}] pid={run['pid']} status={status}")
    log_path = Path(run["log_path"])
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in lines[-10:]:
            print(line)



[cicd_train_bone_fracture_detect_fracture_only_f1_klguard] pid=72973 status=running


## 10) Wait For The Current Detect Probe To Finish

This blocks until the launched detect process exits, then closes the log handle.


In [96]:
if "current_detect_runs" not in globals() or not current_detect_runs:
    raise RuntimeError("Launch the current detect probe first.")

for run in current_detect_runs:
    rc = run["process"].wait()
    run["log_handle"].close()
    print(f"finished {run['name']} with rc={rc}")


finished cicd_train_bone_fracture_detect_fracture_only_f1_klguard with rc=1


## 7) Optional HF Push

Run this if you want to publish the normalized dataset and then train by `dataset_name` instead of `dataset_path`.


In [97]:
!python build_bone_fracture_hf_dataset.py   --config configs/build_bone_fracture_hf_dataset_default.json   --env-file .env.staging   --push-to-hub maxs-m87/bone_fracture_point_v1


using local COCO export at /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture/raw_dataset/bone fracture.coco
Saving the dataset (1/1 shards): 100%|█| 369/369 [00:00<00:00, 3596.66 examples/
Saving the dataset (1/1 shards): 100%|█| 44/44 [00:00<00:00, 2780.03 examples/s]
saved normalized dataset to /Users/maxs/Documents/Repos/MD/MD_RL_Pipe/RL_amazon_logo/bone_fracture/outputs/maxs-m87_bone_fracture_point_v1 (train=369, validation=45, test=44)
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Map: 100%|██████████████████████████| 369/369 [00:00<00:00, 15822.55 examples/s]

Creating parquet from Arrow format: 100%|█████████| 1/1 [00:00<00:00, 71.66ba/s]
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...0gn/T/tmpx_9xq5n6.parquet:  22%|███           | 3.66MB / 16.6MB            

Processing Fil